# BPIC15 RL Pipeline — Concrete Notebook Plan

This notebook is the **implementation roadmap** from process-graph analysis to a first PPO baseline.
It assumes work in `process_graph.ipynb` is complete through:
- data loading (`logs`, `MUNICIPALITIES`)
- DFGs (`dfgs`)
- temporal activity ranks (`act_ranks`)
- cross-municipality transition comparisons

---

## 0) Scope, assumptions, and outputs
**Goal:** Build a reproducible baseline pipeline for case-level decision optimization.

**Primary inputs**
- `./dataset/BPIC15_Municipality{1..5}.jsonocel`
- optional carry-over artifacts from `process_graph.ipynb` (edge frequencies, `act_ranks`)

**Primary outputs (write to `./output`)**
1. `case_step_features.parquet`
2. `transition_stats.csv`
3. `duration_stats.csv`
4. `valid_action_space.csv` (or refreshed version)
5. `sim_calibration.json`
6. `sim_validation_report.csv`
7. trained PPO checkpoint + metrics CSV

**Acceptance criteria**
- Real vs simulated distributions are close on core metrics
- PPO baseline beats simple heuristic baseline on at least one held-out split

---

## 1) Recreate and freeze graph-derived priors (from process_graph)
**Why:** Downstream feature engineering needs stable stage/order signals and branch hints.

**Cells to add**
1. Load logs and rebuild minimal shared objects (`logs`, `dfgs`, `act_ranks`)
2. Compute shared backbone transitions (intersection / high-support edges)
3. Compute branch/rare transitions (low-support or municipality-specific)
4. Persist priors to disk (`graph_priors.json`)

**Checks**
- At least one non-empty backbone set
- Rare transitions include meaningful deviations (not only noise)

---

## 2) Build the case-step feature table (one row per event-in-case)
**Why:** This is the RL state table.

**Target schema (minimum)**
- `municipality`, `case_id`, `event_id`, `timestamp`
- `activity`, `prev_activity`, `next_activity`
- `step_index`, `trace_length`, `stage_rank`
- `time_since_case_start_hours`, `time_since_prev_hours`
- `rework_count_activity`, `seen_activity_before`
- branch flags: `is_refusal_path`, `is_suspension_path`, `is_completeness_path`
- outcome helpers: `is_terminal_event`, `case_completed`

**Cells to add**
1. Build `Application -> ordered events` extractor
2. Row-expansion function per case
3. Concatenate all municipalities into one dataframe
4. Type-cleaning + null handling + sort keys
5. Save `case_step_features.parquet`

**Checks**
- No duplicate key on (`municipality`,`case_id`,`event_id`)
- Time deltas are non-negative
- `step_index` spans `0..trace_length-1` per case

---

## 3) Compute duration and branch statistics for simulator calibration
**Why:** Simulator parameters must be learned from data, not guessed.

**Cells to add**
1. Transition probabilities by municipality and global
2. Activity duration distributions (median, IQR, log-mean if skewed)
3. Branch probabilities from designated decision activities
4. Case-arrival proxy from first-event timestamps
5. Persist as `transition_stats.csv`, `duration_stats.csv`, `sim_calibration.json`

**Checks**
- Transition rows sum to ~1 per source activity
- No empty duration model for frequent activities

---

## 4) Define a small, explicit action space
**Why:** PPO requires clear, enforceable decisions.

**Initial actions**
- `continue_normal_path`
- `prioritize_case`
- `request_missing_info`
- `send_to_review`
- `escalate`
- `close_case`

**Cells to add**
1. Action dictionary and integer mapping
2. State-dependent validity rules (action mask function)
3. Export/refresh `valid_action_space.csv` with rule descriptions

**Checks**
- Every state has at least one valid action
- Invalid actions are deterministic from state features

---

## 5) Specify reward function (baseline)
Use a simple linear reward:

$$
r_t = -\alpha\,\text{delay}_t - \beta\,\text{rework}_t - \delta\,\text{invalid}_t + \gamma\,\text{completion}_t
$$

**Cells to add**
1. Parameter block (`alpha, beta, delta, gamma`)
2. Reward component function returning per-step breakdown
3. Sanity plots of reward distributions by municipality

**Checks**
- Completed cases have higher cumulative reward than stalled cases (median)
- Invalid-action penalty dominates tiny delay differences

---

## 6) Build a calibrated SimPy environment
**Why:** PPO needs many trajectories; simulator must mimic real workflow behavior.

**Cells to add**
1. Case generator using arrival-rate estimates
2. Event progression using transition probabilities + duration model
3. Hook for agent action to influence routing/priority
4. Episode termination and logging to dataframe

**Simulator output schema**
- case trace table compatible with `case_step_features` core columns
- episode summary table: duration, loops, terminal status, action counts

**Checks**
- No impossible transitions outside allowed transition table
- Episode always terminates within max step cap

---

## 7) Validate simulator against real logs (must pass before PPO)
**Comparison metrics**
1. Trace length distribution
2. Activity frequency distribution
3. Transition frequency matrix distance
4. Case duration distribution
5. Loop/rework frequency

**Cells to add**
1. Metric computation real vs simulated
2. Side-by-side plots
3. Summary table + pass/fail thresholds
4. Save `sim_validation_report.csv`

**Suggested thresholds (baseline)**
- Relative error < 20% for top activities
- Duration median error < 25%
- Loop frequency error < 20%

---

## 8) Create RL environment wrapper (Gymnasium-style)
**Observation**
- Manual feature vector from case-step row + small history signals

**Action**
- Discrete index over action space with action masking

**Episode end**
- Case completed, dropped, or max-steps reached

**Cells to add**
1. `Env` class (`reset`, `step`, `action_mask`)
2. Vectorized observation builder
3. Random-policy smoke test (10–20 episodes)

**Checks**
- Shapes/dtypes stable
- No NaN in observations/rewards
- Mask blocks invalid actions consistently

---

## 9) Train baseline PPO
**Cells to add**
1. Train/validation split strategy (e.g., municipality hold-out)
2. PPO config (small MLP, conservative learning rate)
3. Training loop + checkpoints
4. Evaluation against heuristic policy

**Minimum experiment matrix**
- Exp A: train on 1–4, test on 5
- Exp B: rotate held-out municipality
- Exp C: synthetic variant stress test

**Outputs**
- `ppo_metrics.csv`
- `ppo_checkpoint/`
- evaluation summary table

---

## 10) Managerial Insights Extraction and Dashboard Presentation
**Why:** Convert technical RL results into actionable business policies and visualizations.

**Workflow**
1. **Clustering analysis**: Group learned policies by decision context
   - Extract decision patterns from trained PPO embeddings/activations
   - Cluster cases by similar optimal action sequences
2. **Rule extraction**: Convert policy clusters into interpretable if-then rules
   - Decision tree fitting on clustered action patterns
   - Rule simplification and validation
3. **Classification into business categories**
   - Task Assignment Patterns (who should handle the case)
   - Sequencing Decisions (activity ordering)
   - Resource Allocation Strategies (priority, queue management)
4. **Managerial language mapping**
   - Template-based or LLM-assisted conversion to business language
   - Map technical features to business domain terms
5. **Dashboard creation**
   - Policy visualization by municipality
   - KPI comparison: real vs optimized
   - Decision tree diagrams for each policy cluster
   - Recommendation engine for case routing

**Cells to add**
1. Policy clustering from trained agent behavior
2. Rule extraction and simplification
3. Business category classification
4. Managerial language translation (template or LLM-based)
5. Interactive/static dashboard generation
6. Export actionable policy documents (PDF/JSON)

**Outputs**
- `policy_clusters.json` (cluster definitions and rules)
- `business_policies.csv` (managerial language rules)
- `dashboard.html` or dashboard notebook
- `recommendations_engine.json` (rule engine for new cases)

**Checks**
- Rules are interpretable (max 2-3 conditions per rule)
- Coverage: >80% of cases match at least one rule
- Business stakeholders can understand and validate rules

---

## 11) Reporting and decision gate for GNN
**Cells to add**
1. Consolidated results table
2. Error analysis: where policy fails (branch type/stage)
3. Decision gate: only proceed to GNN if transfer/generalization plateaus

**Proceed to GNN only if**
- PPO baseline is stable
- Simulator validation is acceptable
- Transfer gap remains significant
- Managerial insights are validated by domain experts

---

## 12) Suggested notebook cell order (copy into implementation notebook)
1. Setup/imports/config
2. Load OCEL + reusable helpers
3. Graph priors extraction
4. Case-step feature engineering
5. Stats for calibration
6. Action space + reward
7. SimPy simulator
8. Simulator validation
9. Gym env wrapper
10. PPO training
11. Evaluation + report

---

## Definition of done (for this phase)
- Feature table and calibration artifacts are persisted
- Simulator validated against real data with documented errors
- PPO baseline trained and evaluated on held-out municipality
- Clear go/no-go recommendation for adding GNN embeddings next

## **CQL (Conservative Q-Learning) - Detailed Overview**

**What is CQL?**
- **Not** a pre-trained model—it's an **offline RL algorithm** that learns from your historical municipal data (logs)
- Published 2020 by Kumar et al. as a solution to the "offline RL problem"
- Available in Stable-Baselines3 via `sb3-contrib`

---

## **How CQL Works (Simple Explanation)**



In [ ]:
Traditional RL:  Agent explores → gets rewards → learns
Offline RL (CQL): Agent learns ONLY from historical data → stays conservative



CQL adds a **pessimistic penalty** to Q-learning:
1. Learns Q-values from your municipal logs (like your BC does, but via RL)
2. **Key difference**: Penalizes actions it hasn't seen in the data heavily
3. Result: Policy stays close to observed expert behavior (safe) but optimizes where data allows

**Why CQL for this comparison:**
- Learns from same historical municipal data as your BC
- Can optimize beyond pure imitation (unlike BC)
- Stays safe because it's conservative (won't break constraints badly)
- Direct competitor to BC approach

---

## **Training Time Requirements**

| Component | Time | Notes |
|---|---|---|
| **BC Model** | 5-15 min | Fast (supervised learning) |
| **CQL Training** | 30-120 min | Slower (RL exploration on logged data) |
| **Evaluation (both)** | 10-20 min | Rollouts on test cases |
| **Total Comparison** | **1.5-3 hours** | Depending on data size & hyperparams |

**Factors affecting CQL speed:**
- Number of historical cases (more = slower)
- Action space size (resource allocation dimensions)
- Episode length (steps to process each case)
- Network architecture complexity

---

## **Implementation Approach**

Here's the workflow:

### **Step 1: Prepare Offline Dataset**


In [ ]:
# Convert your municipal logs to RL format
observations = []  # states from cases
actions = []       # resource assignments (what actually happened)
rewards = []       # KPI improvements (duration_reduced, etc)
terminals = []     # case completion

# Package into OfflineDataset



### **Step 2: Train CQL**


In [ ]:
from sb3_contrib import CQL

# Use your historical data
cql_model = CQL(
    "MlpPolicy",
    your_env,
    learning_rate=1e-4,
    buffer_size=len(offline_data),  # Fixed size (no new data)
    batch_size=256,
)

cql_model.learn(total_timesteps=100_000)  # Much shorter than PPO



### **Step 3: Compare Against BC**


In [ ]:
metrics = {
    'BC_duration': evaluate_bc_model(bc_policy),
    'CQL_duration': evaluate_cql_model(cql_model),
    'FCFS_duration': evaluate_fcfs_baseline(),
    'Greedy_duration': evaluate_greedy_baseline(),
}



---

## **CQL vs BC: Key Differences**

| Aspect | BC (Your Current) | CQL |
|---|---|---|
| **Training data** | Historical actions (what happened) | States, actions, rewards (why it happened) |
| **Optimization** | Minimize action prediction error | Maximize expected reward from data |
| **Risk** | Conservative (copies exactly) | Slightly more aggressive (optimizes within data support) |
| **When it's better** | Data is high-quality expert | Data has suboptimal decisions you can improve on |
| **When it fails** | Data distribution shifts | Too conservative, stays at expert level |

---

## **Practical Timeline for Your Comparison**

**Assuming BPIC dataset (5 municipalities, ~10k cases total):**



In [ ]:
Monday:
  - 10 min: Prepare offline RL dataset from your logs
  - 20 min: Set up CQL environment wrapper
  - 90 min: Train CQL (overnight if you want)

Tuesday:
  - 30 min: Run evaluation on test set (compare BC vs CQL vs FCFS)
  - 20 min: Generate comparison plots/tables
  - 10 min: Write results summary
  
Total: 3-4 hours wall-clock time (mostly training)



**Can you speed it up?**

- ✅ **Reduce offline data**: Sample 3000 cases instead of 10k → 50% faster training
- ✅ **Smaller network**: 64-dims instead of 256-dims → 30% faster  
- ✅ **Fewer timesteps**: 50k instead of 100k → 50% faster (less convergence)
- ❌ **GPU won't help much**: CQL is memory-bound, not compute-bound

---

## **My Recommendation: Do This Comparison**

**Reasons:**
1. **Complementary to BC**: Shows if RL optimization beats pure imitation
2. **Quick to implement**: ~2 hours total effort (you have the infrastructure)
3. **Strong publication storyline**: 
   - *"BC learns what experts do, CQL learns what optimizes rewards—but BC wins because constraints matter more than pure metric optimization in municipal settings"* (or vice versa)
4. **Reuses your data**: Your calibration data becomes the offline dataset

**Realistic outcome:**
- CQL probably performs **similar or slightly worse than BC** in bureaucratic workflows (because expert decisions are already pretty good, and CQL gets conservative)
- BC probably performs **much better than FCFS/Greedy** (validates your approach)

---

## **Quick Implementation: Want Me To Add CQL To Your Notebook?**

I can add a comparison cell that:
1. ✅ Converts your municipal logs to CQL offline dataset  
2. ✅ Trains CQL model in ~60-90 min
3. ✅ Evaluates both BC and CQL on same test cases
4. ✅ Generates side-by-side metrics table

Would you like me to implement this? It would be a new section in your step8_step9_colab_combined.ipynb that runs after your PPO training completes.